# Alex Open Microwave — GR00T N1.6 Fine-tuning (Colab)

Fine-tune [GR00T N1.6](https://github.com/NVIDIA/Isaac-GR00T) on an Alex ability-hands LeRobot dataset.

**Before opening this notebook:**
1. On your Arena machine, convert HDF5 → LeRobot (`alex_open_microwave_config.yaml`).
2. Upload the `lerobot/` folder to Google Drive (e.g. `MyDrive/alex_demo_generated/lerobot/`).
3. Select a **GPU** runtime (A100 40GB recommended). Training takes several hours.

Evaluation in Isaac Sim still runs on your local Arena machine after you download checkpoints.

In [ ]:
# Runtime check
!nvidia-smi

import torch
assert torch.cuda.is_available(), "Switch Runtime → GPU (A100 recommended)"

In [ ]:
# --- Configuration ---
HF_DATASET_ID = "H2Ozone/alex_microwave"  # HuggingFace dataset repo
DATASET_PATH  = "/content/alex_microwave" # where the dataset will be downloaded

# Checkpoints: change to a Drive path (e.g. /content/drive/MyDrive/...) if you mount Drive below
OUTPUT_DIR    = "/content/alex_gr00t_checkpoints"

ARENA_REPO    = "/content/IsaacLab-Arena"
ISAAC_GR00T_DIR = "/content/Isaac-GR00T"

In [ ]:
# (Optional) Mount Google Drive — only needed if you set OUTPUT_DIR to a Drive path above
# from google.colab import drive
# drive.mount("/content/drive")

In [ ]:
# Download dataset from HuggingFace
!pip install -q huggingface_hub

from huggingface_hub import snapshot_download

snapshot_download(
    repo_id=HF_DATASET_ID,
    repo_type="dataset",
    local_dir=DATASET_PATH,
)
print(f"Dataset downloaded to {DATASET_PATH}")

In [ ]:
# Clone Isaac-GR00T (training) and IsaacLab-Arena (Alex modality config)
%cd /content

!git clone --recurse-submodules https://github.com/NVIDIA/Isaac-GR00T
# Use your fork/branch if alex_data_config.py is not on upstream main yet:
!git clone https://github.com/isaac-sim/IsaacLab-Arena.git

MODALITY_CONFIG = f"{ARENA_REPO}/isaaclab_arena_gr00t/embodiments/alex/alex_data_config.py"
assert __import__("os").path.isfile(MODALITY_CONFIG), f"Missing {MODALITY_CONFIG}"

In [ ]:
# Install GR00T (uv). First run can take 15–30+ minutes (flash-attn build).
%cd /content/Isaac-GR00T

!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]

!uv sync --python 3.10
!uv pip install -e .

In [ ]:
# Verify dataset layout
import os
for sub in ["meta", "data", "videos"]:
    p = os.path.join(DATASET_PATH, sub)
    assert os.path.isdir(p), f"Missing {p} — dataset may not be in LeRobot format"
print("Dataset OK:", DATASET_PATH)

In [ ]:
# Fine-tune (single A100). Lower GLOBAL_BATCH_SIZE to 4 if OOM.
import os
import subprocess

GLOBAL_BATCH_SIZE = 8
MAX_STEPS = 30000
SAVE_STEPS = 5000

os.makedirs(OUTPUT_DIR, exist_ok=True)

cmd = [
    "uv", "run", "python", "gr00t/experiment/launch_finetune.py",
    "--dataset-path", DATASET_PATH,
    "--output-dir", OUTPUT_DIR,
    "--modality-config-path", MODALITY_CONFIG,
    "--global-batch-size", str(GLOBAL_BATCH_SIZE),
    "--max-steps", str(MAX_STEPS),
    "--num-gpus", "1",
    "--save-steps", str(SAVE_STEPS),
    "--save-total-limit", "5",
    "--base-model-path", "nvidia/GR00T-N1.6-3B",
    "--no-tune-llm",
    "--tune-visual",
    "--tune-projector",
    "--tune-diffusion-model",
    "--dataloader-num-workers", "2",
    "--embodiment-tag", "NEW_EMBODIMENT",
    "--color-jitter-params", "brightness", "0.3", "contrast", "0.4", "saturation", "0.5", "hue", "0.08",
]

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["PATH"] = "/root/.local/bin:" + env.get("PATH", "")

print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=ISAAC_GR00T_DIR, env=env, check=True)

## After training

Checkpoints are under `OUTPUT_DIR` (e.g. `checkpoint-5000`, `checkpoint-30000`).

If you saved to `/content/`, download the final checkpoint before the session ends:
```python
from google.colab import files
import shutil
shutil.make_archive("/content/checkpoint_final", "zip", OUTPUT_DIR)
files.download("/content/checkpoint_final.zip")
```

Copy the checkpoint to your Arena machine and serve it with the GR00T policy server.
Use **`NEW_EMBODIMENT`** and the same Alex modality layout (`action_horizon=16`, stereo ZED keys).

Local (non-Colab) equivalent:
```bash
export DATASET_PATH=/tmp/alex_demo_generated/lerobot
export OUTPUT_DIR=~/models/alex_open_microwave_finetune
bash isaaclab_arena_gr00t/training/alex_finetune_single_gpu.sh
```